# Assignment B: Reproducible Segmentation Experiment on Kvasir-SEG
Binary polyp segmentation with a small U-Net. One controlled change: a **global-pooling context branch** in the bottleneck.

## Step 1: Environment and data contract

In [ ]:
import os, io, json, glob, random, time, zipfile, platform
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

# All generated artifacts go here. On Kaggle this is /kaggle/working (the only folder kept as notebook output).
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./artifacts")
os.makedirs(OUT_DIR, exist_ok=True)
OUT = lambda name: os.path.join(OUT_DIR, name)
print("Artifact folder:", OUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

### Locate and unzip the uploaded dataset

In [ ]:
# Works on Colab (uploaded zip) and Kaggle (dataset added via "Add Input")
if os.path.isdir("/kaggle/input"):
    EXTRACT_DIR = "/kaggle/input"          # Kaggle already extracts datasets
else:
    cands = [p for p in glob.glob("/content/*.zip") + glob.glob("./*.zip") if "kvasir" in p.lower()]
    assert cands, "Upload kvasir-seg.zip to /content first"
    EXTRACT_DIR = "/content/kvasir_data"
    if not os.path.isdir(EXTRACT_DIR):
        with zipfile.ZipFile(cands[0]) as z:
            z.extractall(EXTRACT_DIR)

# Find the images/ and masks/ folders wherever they ended up
img_dirs, mask_dirs = [], []
for root, dirs, _ in os.walk(EXTRACT_DIR):
    for d in dirs:
        if d.lower() == "images": img_dirs.append(os.path.join(root, d))
        if d.lower() == "masks":  mask_dirs.append(os.path.join(root, d))
# if other datasets are attached, keep only the Kvasir one
pick = lambda L: [p for p in L if "kvasir" in p.lower()] or L
img_dirs, mask_dirs = pick(img_dirs), pick(mask_dirs)
assert len(img_dirs) == 1 and len(mask_dirs) == 1, f"Ambiguous dataset folders: {img_dirs} {mask_dirs}"
img_dir, mask_dir = img_dirs[0], mask_dirs[0]
print("Images:", img_dir)
print("Masks :", mask_dir)

stem = lambda p: os.path.splitext(os.path.basename(p))[0]
img_files  = {stem(p): p for p in glob.glob(img_dir + "/*")  if p.lower().endswith((".jpg", ".jpeg", ".png"))}
mask_files = {stem(p): p for p in glob.glob(mask_dir + "/*") if p.lower().endswith((".jpg", ".jpeg", ".png"))}
names = sorted(set(img_files) & set(mask_files))
print("Image/mask pairs:", len(names), "| unmatched images:", len(img_files) - len(names),
      "| unmatched masks:", len(mask_files) - len(names))

EXPECTED_PAIRS = 1000   # Kvasir-SEG ships exactly 1000 image/mask pairs
assert len(names) == len(img_files) == len(mask_files) == EXPECTED_PAIRS, (
    f"Dataset incomplete or wrong: {len(names)} matched pairs, {len(img_files)} images, {len(mask_files)} masks "
    f"(expected exactly {EXPECTED_PAIRS} of each, 0 unmatched). Re-add/re-upload the Kvasir-SEG input.")

**Data contract** (Kvasir-SEG, 1000 colonoscopy images with polyp masks)

| Item | Value |
|---|---|
| Task | Binary semantic segmentation |
| Class IDs | 0 = background (normal mucosa), 1 = polyp |
| Ignored label | 255. Masks are stored as JPEG, so edges contain compression-blurred grey values. Mask pixels with value <= 50 become 0, >= 200 become 1, and everything in between becomes 255 and is excluded from loss and metrics |
| Splits | Kvasir-SEG ships no official split, so a seeded random 80/10/10 split is made (train/val/test), saved to `split_indices.json` |
| Image transform | Resize to 256x256, bilinear, scaled to [0,1] |
| Mask transform | Resize to 256x256, nearest-neighbour (keeps labels discrete), then thresholded as above |
| Augmentation | Random horizontal and vertical flip, training split only |

In [ ]:
IMAGE_SIZE = 256
LOW_T, HIGH_T = 50, 200   # grey-value thresholds for the mask

def load_pair(n):
    im = Image.open(img_files[n]).convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    mk = Image.open(mask_files[n]).convert("L").resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)
    im = np.asarray(im, dtype=np.uint8)
    mk = np.asarray(mk, dtype=np.uint8)
    out = np.full(mk.shape, 255, dtype=np.uint8)
    out[mk <= LOW_T] = 0
    out[mk >= HIGH_T] = 1
    return im, out

pairs = [load_pair(n) for n in names]
X = np.stack([p[0] for p in pairs])          # (N, H, W, 3) uint8
Y = np.stack([p[1] for p in pairs])          # (N, H, W)    uint8 in {0, 1, 255}
del pairs
print("X:", X.shape, "| Y:", Y.shape, "| mask values:", np.unique(Y))
print(f"Polyp pixels: {100*(Y==1).mean():.2f}% | background: {100*(Y==0).mean():.2f}% | ignored: {100*(Y==255).mean():.3f}%")

### Seeded train / validation / test split

In [ ]:
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(names))
n_train, n_val = int(0.8 * len(names)), int(0.1 * len(names))
train_idx = perm[:n_train]
val_idx   = perm[n_train:n_train + n_val]
test_idx  = perm[n_train + n_val:]
print("Train / Val / Test:", len(train_idx), len(val_idx), len(test_idx))

with open(OUT("split_indices.json"), "w") as f:
    json.dump({
        "seed": SEED,
        "train": [names[i] for i in train_idx],
        "val":   [names[i] for i in val_idx],
        "test":  [names[i] for i in test_idx],
    }, f, indent=1)

### Look at the data

In [ ]:
fig, ax = plt.subplots(2, 4, figsize=(14, 7))
for k in range(4):
    i = train_idx[k]
    vis = Y[i].astype(np.int64).copy(); vis[vis == 255] = 2
    ax[0, k].imshow(X[i]); ax[0, k].set_title(names[i]); ax[0, k].axis("off")
    ax[1, k].imshow(vis, cmap="viridis", vmin=0, vmax=2); ax[1, k].set_title("mask (yellow = ignored)"); ax[1, k].axis("off")
plt.tight_layout(); plt.show()

## Step 2: Dataset and DataLoaders

In [ ]:
class KvasirDataset(torch.utils.data.Dataset):
    def __init__(self, indices, augment=False):
        self.indices, self.augment = indices, augment
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, k):
        i = self.indices[k]
        img = torch.from_numpy(X[i]).permute(2, 0, 1).float() / 255.0
        msk = torch.from_numpy(Y[i].astype(np.int64))
        if self.augment:
            if random.random() < 0.5:
                img, msk = img.flip(-1), msk.flip(-1)
            if random.random() < 0.5:
                img, msk = img.flip(-2), msk.flip(-2)
        return img, msk

BATCH_SIZE = 16
def make_loaders(seed):
    g = torch.Generator().manual_seed(seed)
    tr = torch.utils.data.DataLoader(KvasirDataset(train_idx, augment=True),  batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=0)
    va = torch.utils.data.DataLoader(KvasirDataset(val_idx),  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    te = torch.utils.data.DataLoader(KvasirDataset(test_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    return tr, va, te

train_loader, val_loader, test_loader = make_loaders(SEED)
imgs, msks = next(iter(train_loader))
print(imgs.shape, msks.shape, torch.unique(msks))
assert imgs.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert set(torch.unique(msks).tolist()) <= {0, 1, 255}

## Step 3: Baseline U-Net and the modified version

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
    def forward(self, x):
        return self.net(x)

class GlobalContext(nn.Module):
    """Image-level pooling branch: pool -> 1x1 conv -> broadcast -> concat -> 1x1 fuse."""
    def __init__(self, c):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c, c, 1), nn.ReLU(inplace=True))
        self.fuse = nn.Sequential(nn.Conv2d(2 * c, c, 1), nn.BatchNorm2d(c), nn.ReLU(inplace=True))
    def forward(self, x):
        g = self.proj(F.adaptive_avg_pool2d(x, 1)).expand_as(x)
        return self.fuse(torch.cat([x, g], dim=1))

class UNetSmall(nn.Module):
    def __init__(self, num_classes=2, base=32, global_pool=False):
        super().__init__()
        b = base
        self.enc1, self.enc2, self.enc3 = DoubleConv(3, b), DoubleConv(b, 2*b), DoubleConv(2*b, 4*b)
        self.bott = DoubleConv(4*b, 8*b)
        self.ctx = GlobalContext(8*b) if global_pool else nn.Identity()   # <- the ONLY difference
        self.pool = nn.MaxPool2d(2)
        self.up3, self.dec3 = nn.ConvTranspose2d(8*b, 4*b, 2, 2), DoubleConv(8*b, 4*b)
        self.up2, self.dec2 = nn.ConvTranspose2d(4*b, 2*b, 2, 2), DoubleConv(4*b, 2*b)
        self.up1, self.dec1 = nn.ConvTranspose2d(2*b, b, 2, 2),   DoubleConv(2*b, b)
        self.head = nn.Conv2d(b, num_classes, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.ctx(self.bott(self.pool(e3)))
        d3 = self.dec3(torch.cat([self.up3(m), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)

count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
set_seed(SEED); m0 = UNetSmall(global_pool=False).to(device)
set_seed(SEED); m1 = UNetSmall(global_pool=True).to(device)
print("Baseline params:", count_params(m0))
print("Modified params:", count_params(m1), "| extra:", count_params(m1) - count_params(m0))
with torch.no_grad():
    print("Output shape:", m1.eval()(imgs.to(device)).shape)
del m0, m1

## Step 4: Training (identical budget for both models)

**Training configuration (identical for both runs):** Adam, lr = 1e-3, batch size 16, cross-entropy with `ignore_index=255`, seed 42, 256x256 input, flips as augmentation. **Stopping rule:** at most 40 epochs, stop early if validation loss has not improved for 7 epochs; the checkpoint with the lowest validation loss is kept.

In [ ]:
LR, MAX_EPOCHS, PATIENCE = 1e-3, 40, 7
criterion = nn.CrossEntropyLoss(ignore_index=255)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    tot, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = criterion(model(x), y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            v = (y != 255).sum().item()
            tot += loss.item() * v; n += v
    return tot / max(n, 1)

def build_model(global_pool):
    """Baseline: seeded init. Modified: starts from the *identical* baseline weights for every
    shared layer; only the new context branch (ctx.*) is freshly initialised."""
    set_seed(SEED)
    base = UNetSmall(global_pool=False).to(device)
    if not global_pool:
        return base
    model = UNetSmall(global_pool=True).to(device)
    res = model.load_state_dict(base.state_dict(), strict=False)
    assert not res.unexpected_keys and all(k.startswith("ctx.") for k in res.missing_keys)
    msd = model.state_dict()
    assert all(torch.equal(v, msd[k]) for k, v in base.state_dict().items()), "shared weights differ"
    print(f"Paired init verified: {len(base.state_dict())} shared tensors identical; new tensors: {len(res.missing_keys)}")
    set_seed(SEED)   # restore RNG state so both runs see identical augmentation draws
    return model

RESUME = True    # resume an interrupted run from <tag>_last.pth if it exists in the artifact folder

def _atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp); os.replace(tmp, path)

def _config(tag, global_pool, n_params):
    return {"tag": tag, "global_pool": global_pool, "seed": SEED, "lr": LR, "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "image_size": IMAGE_SIZE,
            "n_train": int(len(train_idx)), "n_params": int(n_params)}

def _get_rng(tr):
    return {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "loader": tr.generator.get_state()}

def _set_rng(st, tr):
    random.setstate(st["py"]); np.random.set_state(st["np"]); torch.set_rng_state(st["torch"])
    if st["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])
    tr.generator.set_state(st["loader"])

def train_model(tag, global_pool):
    set_seed(SEED)
    tr, va, _ = make_loaders(SEED)
    model = build_model(global_pool)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    cfg = _config(tag, global_pool, count_params(model))
    hist = {"train": [], "val": []}
    best, best_ep, bad, first_epoch, finished = float("inf"), 0, 0, 1, False
    best_path, last_path = OUT(f"{tag}_best.pth"), OUT(f"{tag}_last.pth")

    if RESUME and os.path.exists(last_path):
        ck = torch.load(last_path, map_location="cpu", weights_only=False)   # our own file, contains RNG tuples
        if ck.get("config") == cfg and os.path.exists(best_path):
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"])
            hist, best, best_ep, bad = ck["hist"], ck["best"], ck["best_ep"], ck["bad"]
            first_epoch, finished = ck["epoch"] + 1, ck["finished"]
            _set_rng(ck["rng"], tr)
            print(f"[{tag}] resumed after epoch {ck['epoch']}" + (" (run already finished, skipping training)" if finished else ""))
        else:
            print(f"[{tag}] ignoring resume checkpoint (different config or best checkpoint missing); training from scratch")

    if not finished:
        for ep in range(first_epoch, MAX_EPOCHS + 1):
            t0 = time.time()
            tl, vl = run_epoch(model, tr, opt), run_epoch(model, va)
            hist["train"].append(tl); hist["val"].append(vl)
            print(f"[{tag}] epoch {ep:02d} train {tl:.4f} val {vl:.4f} ({time.time()-t0:.0f}s)")
            stop = False
            if vl < best:
                best, best_ep, bad = vl, ep, 0
                _atomic_save({"model": model.state_dict(), "epoch": ep, "val_loss": vl, "seed": SEED,
                              "global_pool": global_pool, "lr": LR, "batch_size": BATCH_SIZE,
                              "image_size": IMAGE_SIZE}, best_path)
            else:
                bad += 1
                stop = bad >= PATIENCE
            finished = stop or ep == MAX_EPOCHS
            _atomic_save({"model": model.state_dict(), "optimizer": opt.state_dict(), "epoch": ep,
                          "best": best, "best_ep": best_ep, "bad": bad, "hist": hist, "finished": finished,
                          "rng": _get_rng(tr), "config": cfg}, last_path)
            if stop:
                print(f"[{tag}] early stop at epoch {ep}"); break
    hist.update(best_epoch=best_ep, best_val_loss=best, epochs_run=len(hist["train"]))
    json.dump(hist, open(OUT(f"{tag}_history.json"), "w"), indent=1)
    return hist

baseline_hist = train_model("baseline", global_pool=False)
print("Best epoch:", baseline_hist["best_epoch"], "| best val loss:", baseline_hist["best_val_loss"])

## Step 5: Controlled change, global-pooling context branch

In [ ]:
modified_hist = train_model("modified", global_pool=True)
print("Best epoch:", modified_hist["best_epoch"], "| best val loss:", modified_hist["best_val_loss"])

plt.figure(figsize=(9, 5))
plt.plot(baseline_hist["val"], label="Baseline val", marker="o")
plt.plot(modified_hist["val"], label="Global-pool val", marker="s")
plt.plot(baseline_hist["train"], "--", label="Baseline train", alpha=.6)
plt.plot(modified_hist["train"], "--", label="Global-pool train", alpha=.6)
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy loss"); plt.legend(); plt.grid(True)
plt.title("Baseline vs modified"); plt.tight_layout()
plt.savefig(OUT("loss_curves.png"), dpi=160); plt.show()

## Step 6: Evaluation on the held-out test split

In [ ]:
def load_best(tag, global_pool):
    assert os.path.exists(OUT(f"{tag}_best.pth")), f"{tag}_best.pth not found in {OUT_DIR}: run the training cells first"
    ck = torch.load(OUT(f"{tag}_best.pth"), map_location=device, weights_only=True)
    m = UNetSmall(global_pool=global_pool).to(device)
    m.load_state_dict(ck["model"]); return m.eval()

baseline_model = load_best("baseline", False)
modified_model = load_best("modified", True)

@torch.inference_mode()
def evaluate(model, loader):
    conf = torch.zeros(2, 2, dtype=torch.int64)       # rows = truth, cols = prediction
    per_image = []                                    # polyp IoU for each test image
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        valid = y != 255
        conf += torch.bincount((y[valid] * 2 + pred[valid]), minlength=4).reshape(2, 2).cpu()
        for i in range(x.size(0)):
            v = valid[i]
            inter = ((y[i] == 1) & (pred[i] == 1) & v).sum().item()
            union = (((y[i] == 1) | (pred[i] == 1)) & v).sum().item()
            per_image.append(inter / union if union else float("nan"))
    tp = conf.diag().float()
    union = conf.sum(0).float() + conf.sum(1).float() - tp
    iou = tp / union
    dice_polyp = (2 * tp[1] / (conf[:, 1].sum() + conf[1, :].sum()).float()).item()
    return {"background_iou": iou[0].item(), "polyp_iou": iou[1].item(), "miou": iou.mean().item(),
            "polyp_dice": dice_polyp, "confusion": conf.tolist(), "per_image_polyp_iou": per_image}

res_b = evaluate(baseline_model, test_loader)
res_m = evaluate(modified_model, test_loader)

@torch.inference_mode()
def latency(model, runs=100, warmup=20):
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    for _ in range(warmup): model(x)
    if device.type == "cuda": torch.cuda.synchronize()
    ts = []
    for _ in range(runs):
        t = time.perf_counter(); model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        ts.append((time.perf_counter() - t) * 1000)
    return {"median_ms": float(np.median(ts)), "p95_ms": float(np.percentile(ts, 95))}

lat_b, lat_m = latency(baseline_model), latency(modified_model)

import pandas as pd
table = pd.DataFrame({
    "Metric": ["Background IoU", "Polyp IoU", "mIoU", "Polyp Dice", "Parameters", "Median latency (ms)", "P95 latency (ms)"],
    "Baseline":    [res_b["background_iou"], res_b["polyp_iou"], res_b["miou"], res_b["polyp_dice"], count_params(baseline_model), lat_b["median_ms"], lat_b["p95_ms"]],
    "Global-pool": [res_m["background_iou"], res_m["polyp_iou"], res_m["miou"], res_m["polyp_dice"], count_params(modified_model), lat_m["median_ms"], lat_m["p95_ms"]],
})
table.to_csv(OUT("model_comparison.csv"), index=False)
display(table)

### Paired statistical comparison (per-image polyp IoU, same 100 test images)

In [ ]:
a = np.array(res_b["per_image_polyp_iou"]); b = np.array(res_m["per_image_polyp_iou"])
ok = ~(np.isnan(a) | np.isnan(b))
d = (b - a)[ok]                                   # positive = modified better on that image
rs = np.random.RandomState(SEED)
boot = np.array([d[rs.randint(0, len(d), len(d))].mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])
paired = {"n_images": int(ok.sum()), "mean_diff": float(d.mean()), "ci95": [float(lo), float(hi)],
          "modified_better": int((d > 0).sum()), "baseline_better": int((d < 0).sum()), "ties": int((d == 0).sum())}
try:
    from scipy.stats import wilcoxon
    paired["wilcoxon_p"] = float(wilcoxon(d).pvalue)
except Exception as e:
    print("scipy unavailable:", e)
print(paired)
print("CI excludes 0" if (lo > 0 or hi < 0) else "95% CI includes 0: difference not distinguishable from noise on this test set")

### Predictions and failure cases

In [ ]:
def show(indices, fname):
    fname = OUT(fname)
    fig, ax = plt.subplots(len(indices), 4, figsize=(14, 3.6 * len(indices)), squeeze=False)
    for r, k in enumerate(indices):
        i = test_idx[k]
        x = torch.from_numpy(X[i]).permute(2, 0, 1).float().div(255).unsqueeze(0).to(device)
        with torch.inference_mode():
            pb = baseline_model(x).argmax(1)[0].cpu().numpy()
            pm = modified_model(x).argmax(1)[0].cpu().numpy()
        gt = Y[i].astype(np.int64).copy(); gt[gt == 255] = 2
        for c, (im, t, cm) in enumerate([(X[i], f"{names[i]}", None), (gt, "Ground truth", "viridis"),
                                          (pb, "Baseline", "viridis"), (pm, "Global-pool", "viridis")]):
            ax[r, c].imshow(im, cmap=cm, vmin=0 if cm else None, vmax=2 if cm else None)
            ax[r, c].set_title(t); ax[r, c].axis("off")
    plt.tight_layout(); plt.savefig(fname, dpi=160, bbox_inches="tight"); plt.show()

show([0, 1, 2], "predictions.png")

scores = np.array(res_b["per_image_polyp_iou"])
worst = [int(k) for k in np.argsort(np.nan_to_num(scores, nan=9))[:3]]
for k in worst:
    print(f"Test #{k} ({names[test_idx[k]]}): baseline polyp IoU {res_b['per_image_polyp_iou'][k]:.3f} | global-pool {res_m['per_image_polyp_iou'][k]:.3f}")
show(worst, "failure_cases.png")

### Save evidence

In [ ]:
summary = {
    "dataset": "Kvasir-SEG", "images_total": len(names),
    "split": {"train": len(train_idx), "val": len(val_idx), "test": len(test_idx)},
    "classes": {"0": "background", "1": "polyp"}, "ignored_label": 255,
    "mask_thresholds": {"background_le": LOW_T, "polyp_ge": HIGH_T},
    "image_size": IMAGE_SIZE, "seed": SEED, "optimizer": "Adam", "lr": LR, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
    "augmentation": ["hflip p=0.5", "vflip p=0.5"],
    "environment": {"python": platform.python_version(), "torch": torch.__version__,
                    "torchvision": torchvision.__version__,
                    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"},
    "latency_conditions": {"batch_size": 1, "input": [3, IMAGE_SIZE, IMAGE_SIZE], "warmup": 20, "timed_runs": 100, "measures": "forward pass only, synchronised on GPU"},
    "baseline": {k: v for k, v in res_b.items() if k != "per_image_polyp_iou"} | {"parameters": count_params(baseline_model), "latency": lat_b, "best_epoch": baseline_hist["best_epoch"]},
    "modified": {k: v for k, v in res_m.items() if k != "per_image_polyp_iou"} | {"parameters": count_params(modified_model), "latency": lat_m, "best_epoch": modified_hist["best_epoch"]},
}
summary["paired_comparison"] = paired
json.dump(summary, open(OUT("evaluation_results.json"), "w"), indent=1)

## Step 7: Export all artifacts to one ZIP
This cell is self-contained: it only needs the files in `/kaggle/working`, so it also works in a fresh session if the artifact files still exist.
It creates `assignment_b_artifacts.zip` **only if all 10 required files exist**; otherwise it lists what is missing and stops.

In [ ]:
import os, zipfile
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./artifacts")
ZIP_NAME = "assignment_b_artifacts.zip"
REQUIRED = ["baseline_best.pth", "modified_best.pth", "baseline_history.json", "modified_history.json",
            "model_comparison.csv", "evaluation_results.json", "split_indices.json",
            "loss_curves.png", "predictions.png", "failure_cases.png"]
ALLOW_PARTIAL = False   # True = also allow a ZIP of whatever exists (named ..._PARTIAL.zip)
INCLUDE_RESUME = False  # True = also bundle baseline_last.pth / modified_last.pth (~25 MB each, resume state)
if INCLUDE_RESUME:
    REQUIRED += ["baseline_last.pth", "modified_last.pth"]

present = [f for f in REQUIRED if os.path.isfile(os.path.join(OUT_DIR, f))]
missing = [f for f in REQUIRED if f not in present]
print("Artifact folder:", OUT_DIR)
print("Present:", present)
print("Missing:", missing)

if missing and not ALLOW_PARTIAL:
    raise RuntimeError(f"Not creating ZIP: {len(missing)} required artifact(s) missing: {missing}. "
                       "Run the training/evaluation cells (or re-run the whole notebook) first.")

zip_path = os.path.join(OUT_DIR, ZIP_NAME if not missing else ZIP_NAME.replace(".zip", "_PARTIAL.zip"))
tmp_path = zip_path + ".tmp"
with zipfile.ZipFile(tmp_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in present:
        z.write(os.path.join(OUT_DIR, f), arcname=f)
os.replace(tmp_path, zip_path)

with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None, "ZIP is corrupt"
    inside = {i.filename: i.file_size for i in z.infolist()}
assert set(inside) == set(present), f"ZIP contents differ from expected: {set(present) ^ set(inside)}"
for f in present:
    assert inside[f] == os.path.getsize(os.path.join(OUT_DIR, f)), f"size mismatch: {f}"

print("\nZIP written and verified:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
for f in present:
    print(f"  included: {f}  ({inside[f]/1e3:.0f} KB)")
for f in missing:
    print(f"  MISSING : {f}")
print("\nNEXT: files in /kaggle/working are only kept if this run is committed (Save Version > Save & Run All) - "
      "download the ZIP from the version's Output tab before you leave.")